# Next Word Prediction Using LSTM and GRU

## Data Collection

In [ ]:
import nltk
nltk.download('gutenberg')
from nltk.corpus import gutenberg
import pandas as pd
data=gutenberg.raw('shakespeare-hamlet.txt')
with open('hamlet.txt','w') as file:
    file.write(data)


## Data Preprocessing

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
with open('hamlet.txt','r') as file:
    text=file.read().lower()
tokenizer=Tokenizer()
tokenizer.fit_on_texts([text])
total_words=len(tokenizer.word_index)+1
total_words


### Tokenizer Word Index

In [ ]:
tokenizer.word_index


## Create Input Sequences

In [ ]:
input_sequences=[]
for line in text.split('\n'):
    token_list=tokenizer.texts_to_sequences([line])[0]
    for i in range(1,len(token_list)):
        n_gram_sequence=token_list[:i+1]
        input_sequences.append(n_gram_sequence)


### Input Sequences

In [ ]:
input_sequences


## Pad Sequences

In [ ]:
max_sequence_len=max([len(x) for x in input_sequences])
max_sequence_len


### Padded Input Sequences

In [ ]:
input_sequences=np.array(pad_sequences(input_sequences,maxlen=max_sequence_len,padding='pre'))
input_sequences


## Create Predictors and Label

In [ ]:
import tensorflow as tf
x,y=input_sequences[:,:-1],input_sequences[:,-1]


### Predictors

In [ ]:
x


### Labels

In [ ]:
y


## One-Hot Encode Labels

In [ ]:
y=tf.keras.utils.to_categorical(y,num_classes=total_words)
y


## Split the Data into Training and Testing Sets

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)


## Define Early Stopping

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
early_stopping_lstm = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
early_stopping_gru = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)


## LSTM Model

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,Dropout,GRU
lstm_model=Sequential()
lstm_model.add(Embedding(total_words,100,input_length=max_sequence_len-1))
lstm_model.add(LSTM(150,return_sequences=True))
lstm_model.add(Dropout(0.2))
lstm_model.add(LSTM(100))
lstm_model.add(Dense(total_words,activation="softmax"))
lstm_model.compile(loss="categorical_crossentropy",optimizer='adam',metrics=['accuracy'])
lstm_model.summary()


## Train the LSTM Model

In [ ]:
history_lstm=lstm_model.fit(x_train,y_train,epochs=50,validation_data=(x_test,y_test),verbose=1,callbacks=[early_stopping_lstm])


## GRU Model

In [ ]:
gru_model=Sequential()
gru_model.add(Embedding(total_words,100,input_length=max_sequence_len-1))
gru_model.add(GRU(150,return_sequences=True))
gru_model.add(Dropout(0.2))
gru_model.add(GRU(100))
gru_model.add(Dense(total_words,activation="softmax"))
gru_model.compile(loss="categorical_crossentropy",optimizer='adam',metrics=['accuracy'])
gru_model.summary()


## Train the GRU Model

In [ ]:
history_gru=gru_model.fit(x_train,y_train,epochs=50,validation_data=(x_test,y_test),verbose=1,callbacks=[early_stopping_gru])


## Predict Next Word

In [ ]:
def predict_next_word(model, tokenizer, text, max_sequence_len):
    token_list = tokenizer.texts_to_sequences([text])[0]
    if len(token_list) >= max_sequence_len:
        token_list = token_list[-(max_sequence_len-1):]
    token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
    predicted = model.predict(token_list, verbose=0)
    predicted_word_index = np.argmax(predicted, axis=1)
    for word, index in tokenizer.word_index.items():
        if index == predicted_word_index:
            return word
    return None


## Test Prediction with the LSTM Model

In [ ]:
input_text="To be or not to be"
print(f"Input text:{input_text}")
max_sequence_len=lstm_model.input_shape[1]+1
next_word=predict_next_word(lstm_model,tokenizer,input_text,max_sequence_len)
print(f"Next Word Prediction by LSTM:{next_word}")


## Test Prediction with the GRU Model

In [ ]:
input_text="To be or not to be"
print(f"Input text:{input_text}")
max_sequence_len=gru_model.input_shape[1]+1
next_word=predict_next_word(gru_model,tokenizer,input_text,max_sequence_len)
print(f"Next Word Prediction by GRU:{next_word}")


## Save the Models and Tokenizer

In [ ]:
lstm_model.save("next_word_lstm.h5")
gru_model.save("next_word_gru.h5")
import pickle
with open('tokenizer.pickle','wb') as handle:
    pickle.dump(tokenizer,handle,protocol=pickle.HIGHEST_PROTOCOL)


## Test Prediction with Both Models

In [ ]:
input_text="  Barn. Last night of all,When yond same"
print(f"Input text:{input_text}")
max_sequence_len=lstm_model.input_shape[1]+1
next_word_lstm=predict_next_word(lstm_model,tokenizer,input_text,max_sequence_len)
print(f"Next Word Prediction by LSTM:{next_word_lstm}")
next_word_gru=predict_next_word(gru_model,tokenizer,input_text,max_sequence_len)
print(f"Next Word Prediction by GRU:{next_word_gru}")
